## Environment Setup


---



In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!ls /content/drive/MyDrive/Video-Captioning

accuracy_graph.png   images	       predict_realtime.py  requirements.txt
config.py	     loss_graph.png    predict_test.py	    train.py
_config.yml	     model_final       __pycache__	    Your_videos
data		     model.py	       README.md
extract_features.py  predictions.json  references.json


## Import Libraries

In [3]:
import tensorflow as tf
import numpy as np
import cv2

print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)
print("OpenCV:", cv2.__version__)

TensorFlow: 2.20.0
NumPy: 2.0.2
OpenCV: 4.13.0


In [4]:
from tensorflow.keras.applications.resnet50 import ResNet50

model = ResNet50(
    weights="imagenet",
    include_top=False,
    pooling="avg"
)

print(model.output_shape)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
(None, 2048)


## Check the paths

Download the msvd dataset from hugging face and put it in the same folder path as below

Link to the dataset : https://huggingface.co/datasets/friedrichor/MSVD/tree/main

In [5]:
import sys
sys.path.append('/content/drive/MyDrive/Video-Captioning')

import config

print(config.video_dataset_path)
print(config.feature_path)
print(config.temp_path)

/content/drive/MyDrive/MSVD/videos/YouTubeClips
/content/drive/MyDrive/MSVD/features
/content/drive/MyDrive/MSVD/temp


## Check the MSVD folder


In [6]:
import os
#check number of features in the msvd/features
folder_path = '/content/drive/MyDrive/MSVD/features'

try:
    items = os.listdir(folder_path)
    total_items = len(items)
    print(f"Total number of items in the folder: {total_items}")
except FileNotFoundError:
    print("The specified folder path was not found. Please check the spelling.")


Total number of items in the folder: 1971


## Check train is loaded properly

In [8]:
import importlib
import train

importlib.reload(train)

<module 'train' from '/content/drive/MyDrive/Video-Captioning/train.py'>

## Check before training

In [ ]:
import importlib
import train

importlib.reload(train)
importlib.reload(config)

from train import VideoDescriptionTrain
import config

video_to_text = VideoDescriptionTrain(config)

training_list, validation_list = video_to_text.preprocessing()

print("Training captions:", len(training_list))
print("Validation captions:", len(validation_list))

sample_key = next(iter(video_to_text.x_data))

print("Sample video:", sample_key)
print("Feature shape:", video_to_text.x_data[sample_key].shape)

## Train the Model


In [ ]:

import importlib
import train

importlib.reload(train)
importlib.reload(config)

from train import VideoDescriptionTrain
import config

video_to_text = VideoDescriptionTrain(config)
video_to_text.train_model()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, 80, 2048)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, 9, 1500)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 80, 512), │  5,244,928 │ encoder_inputs[0… │
│                     │ (None, 512),      │            │                   │
│                     │ (None, 512)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 9, 512),  │  4,122,624 │ decoder_inputs[0… │
│                     │ (None, 512),      │            │ encoder_lstm[0][… │
│                     │ (None, 512)]      │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bahdanau_attention  │ (None, 9, 512)    │        512 │ decoder_lstm[0][… │
│ (AdditiveAttention) │                   │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 9, 1024)   │          0 │ decoder_lstm[0][… │
│ (Concatenate)       │                   │            │ bahdanau_attenti… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_softmax     │ (None, 9, 1500)   │  1,537,500 │ concatenate_1[0]… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 10,905,564 (41.60 MB)

 Trainable params: 10,905,564 (41.60 MB)

 Non-trainable params: 0 (0.00 B)

...

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ decoder_inputs      │ (None, 9, 1500)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_2       │ (None, 512)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_3       │ (None, 512)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 9, 512),  │  4,122,624 │ decoder_inputs[0… │
│                     │ (None, 512),      │            │ input_layer_2[0]… │
│                     │ (None, 512)]      │            │ input_layer_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_4       │ (None, 80, 512)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bahdanau_attention  │ (None, 9, 512)    │        512 │ decoder_lstm[1][… │
│ (AdditiveAttention) │                   │            │ input_layer_4[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (None, 9, 1024)   │          0 │ decoder_lstm[1][… │
│ (Concatenate)       │                   │            │ bahdanau_attenti… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_softmax     │ (None, 9, 1500)   │  1,537,500 │ concatenate_2[0]… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,660,636 (21.59 MB)

 Trainable params: 5,660,636 (21.59 MB)

 Non-trainable params: 0 (0.00 B)

## Test Predictions on the trained model

In [ ]:
import random
import os
import numpy as np

import importlib
import config
import model
import predict_test

importlib.reload(config)
importlib.reload(model)
importlib.reload(predict_test)

from predict_test import VideoDescriptionInference

video_to_text = VideoDescriptionInference(config)

files = random.sample(
    os.listdir("/content/drive/MyDrive/MSVD/features"),
    10
)

for f in files:
    feature = np.load(
        "/content/drive/MyDrive/MSVD/features/" + f
    )

    caption = video_to_text.greedy_search(feature)

    print("="*60)
    print(f)
    print(caption)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 450ms/step
Qp-k0H93iJE_35_39.avi.npy
a cricket player is playing cricket
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 383ms/step
5x_OGEdO6Z8_0_21.avi.npy
a girl is doing her hair
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 168ms/step
FWzsXeXCwuc_111_116.avi.npy
a man is taking a picture of a bug
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 175ms/step
LuQ0KiMMhoI_49_64.avi.npy
a man is filing papers
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 157ms/step
q9ew_nITQWY_54_62.avi.npy
a man is kicking a soccer
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 250ms/step
mbesJaS6vwg_187_195.avi.npy
a woman is pouring oil into a pot
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 245ms/step
6GJ1DNOGDAM_223_233.avi.npy
a girl is her hair
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 164ms/step
m1NR0uNNs5Y_88_94.avi.npy
a woman is peeling a potato
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 167ms/step
YmXCfQm0_CA_140_156.avi.npy
a man is pulling a dead deer
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 174ms/step
PBHZtoygOYg_450_465.avi.npy
a man is shooting a gun


In [ ]:
!pip install nltk rouge-score -q

  Preparing metadata (setup.py) ... done


**GENERATE PREDICTIONS FOR ALL TEST VIDEOS**



---




In [9]:
import importlib

import config
import model
import predict_test

importlib.reload(config)
importlib.reload(model)
importlib.reload(predict_test)

print("Reload complete")

Reload complete


In [ ]:
print(config.__file__)
print(model.__file__)
print(predict_test.__file__)

from predict_test import VideoDescriptionInference

video_to_text = VideoDescriptionInference(config)

video_to_text.inf_decoder_model.summary()

import numpy as np

feature = np.load(
    "/content/drive/MyDrive/MSVD/features/KrBeBabazDU_15_20.avi.npy"
)

caption = video_to_text.greedy_search(feature)

print(caption)


print("Search type:", config.search_type)
print("Save model path:", config.save_model_path)
print(len(video_to_text.tokenizer.word_index))

/content/drive/MyDrive/Video-Captioning/config.py
/content/drive/MyDrive/Video-Captioning/model.py
/content/drive/MyDrive/Video-Captioning/predict_test.py


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_5       │ (None, None,      │          0 │ -                 │
│ (InputLayer)        │ 1500)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_6       │ (None, 512)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_7       │ (None, 512)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, None,     │  4,122,624 │ input_layer_5[0]… │
│                     │ 512), (None,      │            │ input_layer_6[0]… │
│                     │ 512), (None,      │            │ input_layer_7[0]… │
│                     │ 512)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_8       │ (None, 80, 512)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ additive_attention… │ (None, None, 512) │        512 │ lstm_1[0][0],     │
│ (AdditiveAttention) │                   │            │ input_layer_8[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, None,      │          0 │ lstm_1[0][0],     │
│ (Concatenate)       │ 1024)             │            │ additive_attenti… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, None,      │  1,537,500 │ concatenate_1[0]… │
│                     │ 1500)             │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,660,636 (21.59 MB)

 Trainable params: 5,660,636 (21.59 MB)

 Non-trainable params: 0 (0.00 B)

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
a girl is talking on the phone
Search type: greedy
Save model path: /content/drive/MyDrive/Video-Captioning/model_final
6556


In [ ]:
print("Tokenizer vocabulary:", len(video_to_text.tokenizer.word_index))
print("Model vocabulary:", config.num_decoder_tokens)

Tokenizer vocabulary: 6556
Model vocabulary: 1500


In [ ]:
#preditcitons
import json
import numpy as np
from predict_test import VideoDescriptionInference
import config

video_to_text = VideoDescriptionInference(config)

with open("/content/drive/MyDrive/MSVD/captions/msvd_test.json") as f:
    test_data = json.load(f)

predictions = {}
references = {}

for idx, item in enumerate(test_data):

    video_id = item["video_id"]

    feature_file = f"/content/drive/MyDrive/MSVD/features/{video_id}.avi.npy"

    feature = np.load(feature_file)

    predicted_caption = video_to_text.greedy_search(feature)

    predictions[video_id] = predicted_caption
    references[video_id] = item["caption"]

    if idx % 50 == 0:
        print(idx, predicted_caption)

print("Finished generating captions")
with open("/content/drive/MyDrive/Video-Captioning/predictions.json", "w") as f:
    json.dump(predictions, f)

with open("/content/drive/MyDrive/Video-Captioning/references.json", "w") as f:
    json.dump(references, f)

print("Saved predictions and references")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 165ms/step
0 a man is doing a board
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step
1/1 ━━━━━━━━━━━━━━━━━━━

In [ ]:
import os

print(os.path.exists("/content/drive/MyDrive/Video-Captioning/predictions.json"))
print(os.path.exists("/content/drive/MyDrive/Video-Captioning/references.json"))

True
True


**SCORE THE MODEL**

In [ ]:
!pip install nltk rouge-score -q

In [ ]:
import json

with open("/content/drive/MyDrive/Video-Captioning/predictions.json") as f:
    predictions = json.load(f)

with open("/content/drive/MyDrive/Video-Captioning/references.json") as f:
    references = json.load(f)

print("Predictions:", len(predictions))
print("References:", len(references))

Predictions: 670
References: 670


# BLEU SCORE

In [ ]:
from nltk.translate.bleu_score import corpus_bleu

references_bleu = []
hypotheses = []

for vid in predictions:

    refs = [r.split() for r in references[vid]]
    hyp = predictions[vid].split()

    references_bleu.append(refs)
    hypotheses.append(hyp)

bleu1 = corpus_bleu(
    references_bleu,
    hypotheses,
    weights=(1,0,0,0)
)

bleu2 = corpus_bleu(
    references_bleu,
    hypotheses,
    weights=(0.5,0.5,0,0)
)

bleu3 = corpus_bleu(
    references_bleu,
    hypotheses,
    weights=(1/3,1/3,1/3,0)
)

bleu4 = corpus_bleu(
    references_bleu,
    hypotheses,
    weights=(0.25,0.25,0.25,0.25)
)

print("BLEU-1 =", round(bleu1,4))
print("BLEU-2 =", round(bleu2,4))
print("BLEU-3 =", round(bleu3,4))
print("BLEU-4 =", round(bleu4,4))

BLEU-1 = 0.7906
BLEU-2 = 0.6379
BLEU-3 = 0.5266
BLEU-4 = 0.4135


# METEOR SCORE

In [ ]:
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.translate.meteor_score import meteor_score
import numpy as np

meteor_scores = []

for vid in predictions:
    refs = [r.split() for r in references[vid]]
    hyp = predictions[vid].split()

    meteor_scores.append(
        meteor_score(refs, hyp)
    )

meteor = np.mean(meteor_scores)

print("METEOR =", round(meteor, 4))

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


METEOR = 0.5753


# ROUGE-L SCORE

In [ ]:
from rouge_score import rouge_scorer
import numpy as np

scorer = rouge_scorer.RougeScorer(
    ['rougeL'],
    use_stemmer=True
)

rouge_scores = []

for vid in predictions:

    pred = predictions[vid]

    best_score = 0

    for ref in references[vid]:

        score = scorer.score(ref, pred)

        best_score = max(
            best_score,
            score["rougeL"].fmeasure
        )

    rouge_scores.append(best_score)

rouge_l = np.mean(rouge_scores)

print("ROUGE-L =", round(rouge_l, 4))

ROUGE-L = 0.6425


In [ ]:
import json

with open("/content/drive/MyDrive/MSVD/captions/msvd_train.json") as f:
    train_data = json.load(f)

with open("/content/drive/MyDrive/MSVD/captions/msvd_test.json") as f:
    test_data = json.load(f)

train_ids = set(item["video_id"] for item in train_data)
test_ids = set(item["video_id"] for item in test_data)

overlap = train_ids.intersection(test_ids)

print("Train videos:", len(train_ids))
print("Test videos:", len(test_ids))
print("Overlap:", len(overlap))

Train videos: 1200
Test videos: 670
Overlap: 0


# Run predict_realtime.py for testing on random videos

- you should put the videos in Your_videos folder
- this will work both on colab and locally, colab will donwload the captioned video in outputs folder

In [9]:
%cd /content/drive/MyDrive/Video-Captioning
!python predict_realtime.py

/content/drive/MyDrive/Video-Captioning
[INFO] Loading models...
2026-07-17 21:34:40.232290: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)

Available videos:

1. Recording 2026-07-18 004333.mp4
2. Recording 2026-07-18 011709.mp4
3. Recording 2026-07-18 024726.mp4
4. Recording 2026-07-18 024907.mp4
5. Recording 2026-07-18 030254.mp4

Select video number (q to quit): 5

Processing: Recording 2026-07-18 030254.mp4
/content/drive/MyDrive/Video-Captioning/Your_videos/Recording 2026-07-18 030254
Processing video /content/drive/MyDrive/Video-Captioning/Your_videos/Recording 2026-07-18 030254.mp4
Video path: /content/drive/MyDrive/Video-Captioning/Your_videos/Recording 2026-07-18 030254.mp4
Exists: True
Opened: True
Frames extracted: 543
Feature shape: (80, 2048)
[CLEANUP] Deleted temp folder: /content/drive/MyDrive/MSVD/temp
Caption : a man is running
Inference time : 36.02 sec

Saved 

In [1]:
%cd /content/drive/MyDrive/Video-Captioning

/content/drive/MyDrive/Video-Captioning
